In [ ]:
# Common imports + DES (Modelica) translator pieces.
import json
import shutil

from geojson_modelica_translator.modelica.modelica_runner import ModelicaRunner
from geojson_modelica_translator.system_parameters.system_parameters import (
    SystemParameters,
)

from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
    copy_activity_to_new_location,
)

# -- Execution mode -----------------------------------------------------------
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
# Change this to keep models/results outside the notebook folder (recommended).
# Examples: "../esbe_2026" (one level up) or "esbe_2026" (inside this folder).
MODEL_OUTPUT_SUBDIR = "../test_activities/esbe_2026"
paths = setup_notebook_paths(analysis_subdir=MODEL_OUTPUT_SUBDIR)
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

In [ ]:
# Copy activity_03/coincident to activity_21/coincident (deep copy of the full folder tree).
source_dir = paths.activity_dir("activity_03") / "coincident"
activity_4_dir = paths.activity_dir("activity_21")
dest_dir = activity_4_dir / "coincident"
copy_activity_to_new_location(source_dir, dest_dir)

# Set up the coincident project from activity_21/coincident
uo_coincident = UO(activity_4_dir, "coincident", template_dir=template_data_dir)

# Weather data is copied over from the activity_03/coincident project. No
# need to update the weather information.

### 5G: CLI-based (this is not recommended)

In [ ]:
# ***** WARNING: This will delete the previous run model *****

# Common paths used by the DES (Modelica) steps below.
# Saved as relative-style paths because they're consumed from within Docker.
scenario_path = uo_coincident.project_path / "baseline_scenario.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# Rebuild sys params as a true 5G file.
sys_param_path_5g = uo_coincident.project_path / "sys_params_5g.json"

sp_5g = SystemParameters()
sp_5g.csv_to_sys_param(
    model_type="time_series",
    sys_param_filename=sys_param_path_5g,
    scenario_dir=uo_coincident.project_path / "run" / "baseline_scenario",
    feature_file=feature_path,
    # Use 5G_ghe because the CLI 5G creator currently requires GHE couplings.
    district_type="5G_ghe",
    # Absolute paths avoid cwd-dependent failures when creating 5G models.
    relative_path=None,
    modelica_load_filename="modelica.mos",
    overwrite=True,
)
print(f"Created sys params: {sys_param_path_5g}")

sys_param_data_5g = json.loads(sys_param_path_5g.read_text())
if "fifth_generation" not in sys_param_data_5g.get("district_system", {}):
    raise ValueError(
        f"Expected fifth_generation section in {sys_param_path_5g}, but it was not found."
    )

# Loop-order fallback is now handled in GMT when _loop_order.json is missing.
des_scenario_name = "five_g_cli"
des_model_path = activity_4_dir / des_scenario_name

# Remove old output so we don't mistake stale files for a successful create.
if des_model_path.exists():
    print(f"Removing existing model directory: {des_model_path}")
    shutil.rmtree(des_model_path)

uo_coincident.des_create(
    sys_param_path_5g, feature_path, des_scenario_name, overwrite=True
)

print(f"Created 5G model at: {des_model_path}")

In [ ]:
# Run the generated 5G CLI model with ModelicaRunner. Full-year run is long.
full_year_run = False
activity_4_dir = paths.activity_dir("activity_21")
des_scenario_name = "five_g_cli"
des_analysis_baseline_dir = activity_4_dir / des_scenario_name

package_mo = des_analysis_baseline_dir / "package.mo"
if not package_mo.exists():
    raise FileNotFoundError(f"Expected package.mo not found: {package_mo}")

# Pick the actual top-level Modelica class produced by des_create.
districts_dir = des_analysis_baseline_dir / "Districts"
model_candidates = [
    "DistrictEnergySystem",  # standard GMT output
    "district",  # legacy/aggregated template naming
    des_scenario_name,
    des_scenario_name.lower(),
]
model_class = None
for candidate in model_candidates:
    if (districts_dir / f"{candidate}.mo").exists():
        model_class = candidate
        break
if model_class is None:
    district_files = (
        sorted(p.name for p in districts_dir.glob("*.mo"))
        if districts_dir.exists()
        else []
    )
    raise FileNotFoundError(
        f"Could not infer district model class in {districts_dir}. Found: {district_files}"
    )

model_name = f"{des_scenario_name}.Districts.{model_class}"
print(f"Model: {model_name}")
print(f"file_to_load={package_mo}")
print(f"run_path={des_analysis_baseline_dir}")


def _show_stdout_log(results_path):
    """Print the full stdout.log (also captures Docker stderr) with size info."""
    log_path = results_path / "stdout.log"
    if not log_path.exists():
        print("  stdout.log: NOT FOUND")
        return
    content = log_path.read_text()
    size = log_path.stat().st_size
    print(f"  stdout.log ({size} bytes):")
    if content.strip():
        print(content)
    else:
        print(
            "  <empty — Docker killed the container before OMC wrote anything.\n"
            "  Common causes: OOM kill (exit 137), insufficient Docker memory, or daemon crash.\n"
            "  Try: Docker Desktop → Settings → Resources → increase Memory to ≥8 GB."
        )


mr = ModelicaRunner()
base_kwargs = {
    "file_to_load": str(package_mo),
    "run_path": des_analysis_baseline_dir,
}

# ── Step 1: compile only ─────────────────────────────────────────────────────
# Isolates build errors from runtime / resource failures.
compile_results_stem = f"{model_name}_results"
compile_results_path = des_analysis_baseline_dir / compile_results_stem
if compile_results_path.exists():
    shutil.rmtree(compile_results_path)

print("\n── Compile only ───────────────────────────────────────────────────────")
compile_success, compile_results_path = mr.run_in_docker(
    "compile", model_name, **base_kwargs
)
print(f"Compile success: {compile_success}")
_show_stdout_log(compile_results_path)

if not compile_success:
    raise RuntimeError(
        "Compilation failed — fix errors above before attempting a simulation run."
    )

# ── Step 2: run only ─────────────────────────────────────────────────────────
print("\n── Simulate only ──────────────────────────────────────────────────────")
fmu_candidates = sorted(des_analysis_baseline_dir.glob("*.fmu"))
if not fmu_candidates:
    raise FileNotFoundError(
        f"No FMU found in {des_analysis_baseline_dir} after successful compile."
    )
fmu_file = fmu_candidates[0]
print(f"FMU: {fmu_file}")

run_results_stem = f"{model_name}_results"
results_path = des_analysis_baseline_dir / run_results_stem
if results_path.exists():
    shutil.rmtree(results_path)
    print(f"Removed existing results directory: {results_path}")

run_kwargs = {
    "file_to_load": str(fmu_file),
    "run_path": des_analysis_baseline_dir,
    "step_size": 300,
}
if full_year_run:
    run_kwargs["stop_time"] = 31_536_000  # full year
else:
    run_kwargs["start_time"] = 5_097_600  # March 1
    run_kwargs["stop_time"] = 6_307_200  # March 14

success, results_path = mr.run_in_docker("run", model_name, **run_kwargs)
print(f"Run success: {success}")
_show_stdout_log(results_path)

if not success:
    raise RuntimeError(
        f"Simulation run failed — see stdout.log above.\nResults dir: {results_path}"
    )

print(f"\n5G results path: {results_path}")